In [1]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
# this should be because we are using .venv module
import sys
sys.path.append('..')

import pandas as pd
from src.paths import TRANSFORMED_DATA_DIR

df = pd.read_parquet(TRANSFORMED_DATA_DIR / 'tabular_data.parquet')
df.head()

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,pickup_location_id,target_ride_next_hour
0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,...,2.0,0.0,1.0,0.0,0.0,0.0,0.0,2022-01-29 00:00:00,1,0.0
1,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2022-01-29 01:00:00,1,0.0
2,0.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,1.0,2.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-29 02:00:00,1,0.0
3,0.0,1.0,1.0,0.0,2.0,0.0,0.0,1.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-29 03:00:00,1,0.0
4,1.0,1.0,0.0,2.0,0.0,0.0,1.0,2.0,1.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022-01-29 04:00:00,1,0.0


In [6]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date=datetime(2022, 6, 1, 0, 0, 0),
    target_column_name='target_ride_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')

X_train.shape=(782280, 674)
y_train.shape=(782280,)
X_test.shape=(1360775, 674)
y_test.shape=(1360775,)


### Hyperparameter tuning with Optuna

In [7]:
import numpy as np
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
import optuna

from src.model import get_pipeline

def objective(trial: optuna.trial.Trial) -> float:
    """
    Given a set of hyper-parameters, it trains a model and computes an average
    validation error based on a TimeSeriesSplit
    """
    # pick hyper-parameters
    hyperparams = {
        "metric": 'mae',
        "verbose": -1,
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.2, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 100),   
    }
       
    tss = KFold(n_splits=3)
    scores = []
        
    for train_index, val_index in tss.split(X_train):

        # split data for training and validation
        X_train_, X_val_ = X_train.iloc[train_index, :], X_train.iloc[val_index,:]
        y_train_, y_val_ = y_train.iloc[train_index], y_train.iloc[val_index]
        
        # train the model
        pipeline = get_pipeline(**hyperparams)
        pipeline.fit(X_train_, y_train_)
        
        # evaluate the model
        y_pred = pipeline.predict(X_val_)
        mae = mean_absolute_error(y_val_, y_pred)

        scores.append(mae)
   
    # Return the mean score
    return np.array(scores).mean()

In [8]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

[I 2024-11-15 17:47:27,374] A new study created in memory with name: no-name-edcdeffa-0e10-4a32-84ed-e4df8bcb36c7
[I 2024-11-15 17:48:33,142] Trial 0 finished with value: 2.507832732891762 and parameters: {'num_leaves': 186, 'feature_fraction': 0.4128335817806866, 'bagging_fraction': 0.552865894709404, 'min_child_samples': 49}. Best is trial 0 with value: 2.507832732891762.
[I 2024-11-15 17:49:30,610] Trial 1 finished with value: 2.511588811195335 and parameters: {'num_leaves': 144, 'feature_fraction': 0.9233652662489433, 'bagging_fraction': 0.7439073402580045, 'min_child_samples': 65}. Best is trial 0 with value: 2.507832732891762.
[I 2024-11-15 17:50:14,897] Trial 2 finished with value: 2.5333779678875543 and parameters: {'num_leaves': 67, 'feature_fraction': 0.645358096741106, 'bagging_fraction': 0.4186030408090523, 'min_child_samples': 11}. Best is trial 0 with value: 2.507832732891762.
[I 2024-11-15 17:50:55,731] Trial 3 finished with value: 2.5689992157134647 and parameters: {'nu

In [9]:
best_params = study.best_trial.params
print(f'{best_params=}')

best_params={'num_leaves': 186, 'feature_fraction': 0.4128335817806866, 'bagging_fraction': 0.552865894709404, 'min_child_samples': 49}


In [10]:
pipeline = get_pipeline(**best_params)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function average_rides_last_4_weeks at 0x11e5fd5a0>)),
                ('temporalfeaturesengineer', TemporalFeaturesEngineer()),
                ('lgbmregressor',
                 LGBMRegressor(bagging_fraction=0.552865894709404,
                               feature_fraction=0.4128335817806866,
                               min_child_samples=49, num_leaves=186))])

In [11]:
predictions = pipeline.predict(X_test)
test_mae = mean_absolute_error(y_test, predictions)
print(f'{test_mae=:.4f}')

test_mae=2.5245


In [12]:
from src.plot import plot_one_sample

plot_one_sample(
    example_id=2979,
    features=X_test,
    targets=y_test,
    predictions=pd.Series(predictions)
)

In [13]:
plot_one_sample(
    example_id=1300,
    features=X_test,
    targets=y_test,
    predictions=pd.Series(predictions)
)